In [ ]:
import os
import shutil
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


DRIVE_IMG = "/content/drive/MyDrive/extracting_slums_from_satellite_imagery/extracting_slums_from_satellite_imagery/images"
DRIVE_LBL = "/content/drive/MyDrive/extracting_slums_from_satellite_imagery/extracting_slums_from_satellite_imagery/labels"
!mkdir -p /content/local_data/images /content/local_data/labels
print("Syncing data to local SSD...")
!cp -r "{DRIVE_IMG}/." /content/local_data/images/
!cp -r "{DRIVE_LBL}/." /content/local_data/labels/
print("Sync Complete.")


Mounted at /content/drive
Syncing data to local SSD...
Sync Complete.


In [ ]:
import glob
import os
from sklearn.model_selection import train_test_split

all_img_paths = sorted(glob.glob("/content/local_data/images/*.tif"))
all_lbl_paths = sorted(glob.glob("/content/local_data/labels/*.tif"))

img_basenames = {os.path.basename(p).split('.')[0] for p in all_img_paths}
lbl_basenames = {os.path.basename(p).split('.')[0] for p in all_lbl_paths}
common_basenames = sorted(list(img_basenames.intersection(lbl_basenames)))

all_img_files = []
all_lbl_files = []
for basename in common_basenames:
    img_path = f"/content/local_data/images/{basename}.tif"
    lbl_path = f"/content/local_data/labels/{basename}.tif"
    if os.path.exists(img_path) and os.path.exists(lbl_path):
        all_img_files.append(img_path)
        all_lbl_files.append(lbl_path)

train_imgs, temp_imgs, train_lbls, temp_lbls = train_test_split(
    all_img_files, all_lbl_files, test_size=0.2, random_state=42
)
val_imgs, test_imgs, val_lbls, test_lbls = train_test_split(
    temp_imgs, temp_lbls, test_size=0.5, random_state=42
)

print(f"Total images: {len(all_img_files)}")
print(f"Training images: {len(train_imgs)}")
print(f"Validation images: {len(val_imgs)}")
print(f"Test images: {len(test_imgs)}")

Total images: 2323
Training images: 1858
Validation images: 232
Test images: 233


In [ ]:
TILE_SIZE = 256
STRIDE = 192   # changed for some overlap; you can set STRIDE = TILE_SIZE for no overlap
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

In [ ]:
import rasterio
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import random
import math

In [ ]:
def select_rgb_bands(img):
    """
    Input: img ndarray with shape (bands, H, W)
    Strategy:
     - if bands >=3: use first 3 bands
     - if bands == 2: replicate 2nd channel to make 3
     - if bands == 1: replicate single channel to (R,G,B)
    If you have a consistent band order with NIR at band 4 and you want (R,G,NIR) or NDVI,
    modify this function accordingly.
    """
    b = img.shape[0]
    if b >= 3:
        return img[:3]
    elif b == 2:
        return np.vstack([img, img[1:2]])
    else:
        return np.vstack([img, img, img])

In [ ]:
def load_tiff_pair(img_path, lbl_path):
    with rasterio.open(img_path) as src:
        img = src.read()  # shape (bands, H, W)
    with rasterio.open(lbl_path) as src:
        mask = src.read(1)
    img = select_rgb_bands(img)
    mask = (mask > 0).astype("uint8")
    return img, mask

In [ ]:
def tile_image(img, mask, tile_size=TILE_SIZE, stride=STRIDE):
    """
    Returns list of (img_tile, mask_tile).
    Ensures the full image is covered by including last tiles at the edges.
    """
    _, H, W = img.shape
    tiles = []
    ys = list(range(0, max(1, H - tile_size + 1), stride))
    xs = list(range(0, max(1, W - tile_size + 1), stride))
    if ys[-1] + tile_size < H:
        ys.append(H - tile_size)
    if xs[-1] + tile_size < W:
        xs.append(W - tile_size)

    for y in ys:
        for x in xs:
            img_t = img[:, y:y+tile_size, x:x+tile_size]
            mask_t = mask[y:y+tile_size, x:x+tile_size]
            # if the tile is smaller (shouldn't for the way we built ys/xs), pad it
            if img_t.shape[1] != tile_size or img_t.shape[2] != tile_size:
                pad_h = tile_size - img_t.shape[1]
                pad_w = tile_size - img_t.shape[2]
                img_t = np.pad(img_t, ((0,0),(0,pad_h),(0,pad_w)), mode='constant')
                mask_t = np.pad(mask_t, ((0,pad_h),(0,pad_w)), mode='constant')
            tiles.append((img_t, mask_t))
    return tiles

In [ ]:
def build_tiles(img_list, lbl_list, min_mask_fraction=0.0005, keep_empty_frac=0.2):
    """
    Build tiles and do optional filtering:
     - min_mask_fraction: if a tile's mask fraction is < this, considered background tile.
     - keep_empty_frac: fraction of background tiles retained (to avoid total removal of negative examples).
    """
    tiles = []
    background_tiles = []
    for img_p, lbl_p in zip(img_list, lbl_list):
        img, mask = load_tiff_pair(img_p, lbl_p)
        ts = tile_image(img, mask)
        for (im_t, m_t) in ts:
            frac = m_t.sum() / (m_t.shape[0] * m_t.shape[1])
            if frac >= min_mask_fraction:
                tiles.append((im_t, m_t))
            else:
                background_tiles.append((im_t, m_t))
    # retain a proportion of background tiles to keep negatives in training
    n_keep = int(len(background_tiles) * keep_empty_frac)
    if n_keep > 0:
        tiles.extend(random.sample(background_tiles, k=min(n_keep, len(background_tiles))))
    else:
        tiles.extend(background_tiles)  # fallback: keep all
    return tiles

train_tiles = build_tiles(train_imgs, train_lbls, min_mask_fraction=0.0002, keep_empty_frac=0.25)
val_tiles   = build_tiles(val_imgs, val_lbls, min_mask_fraction=0.0, keep_empty_frac=1.0)  # keep all for validation
test_tiles  = build_tiles(test_imgs, test_lbls, min_mask_fraction=0.0, keep_empty_frac=1.0)

print("Train tiles:", len(train_tiles))
print("Val tiles:", len(val_tiles))
print("Test tiles:", len(test_tiles))

Train tiles: 4747
Val tiles: 928
Test tiles: 932


In [ ]:
class SlumDataset(Dataset):
    def __init__(self, tiles, training=False):
        self.tiles = tiles
        self.training = training

    def __len__(self):
        return len(self.tiles)

    def random_augment(self, img, mask):
        # img: (3,H,W), mask: (H,W)
        # random horizontal/vertical flips
        if random.random() < 0.5:
            img = np.flip(img, axis=2).copy()
            mask = np.flip(mask, axis=1).copy()
        if random.random() < 0.5:
            img = np.flip(img, axis=1).copy()
            mask = np.flip(mask, axis=0).copy()
        # random 90-degree rotations
        k = random.choice([0,1,2,3])
        img = np.rot90(img, k=k, axes=(1,2)).copy()
        mask = np.rot90(mask, k=k, axes=(0,1)).copy()
        # brightness jitter
        if random.random() < 0.5:
            factor = 0.9 + random.random()*0.2
            img = img * factor
        # gaussian noise
        if random.random() < 0.25:
            noise = np.random.normal(0, 0.01, img.shape)
            img = img + noise
        return img, mask

    def __getitem__(self, idx):
        img, mask = self.tiles[idx]
        # convert to float [0,1]
        img = img.astype(np.float32) / 255.0
        if self.training:
            img, mask = self.random_augment(img, mask)
        # normalize
        mean = np.array(MEAN).reshape(3,1,1)
        std  = np.array(STD).reshape(3,1,1)
        img = (img - mean) / std
        img = torch.tensor(img.copy(), dtype=torch.float32)
        mask = torch.tensor(mask.copy(), dtype=torch.long)  # long for metric computation; we'll cast to float for loss
        return img, mask

train_loader = DataLoader(SlumDataset(train_tiles, training=True), batch_size=8, shuffle=True, num_workers=0, pin_memory=True)
val_loader   = DataLoader(SlumDataset(val_tiles, training=False), batch_size=8, shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(SlumDataset(test_tiles, training=False), batch_size=8, shuffle=False, num_workers=0, pin_memory=True)

print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))

Train batches: 594
Val batches: 116
Test batches: 117


In [ ]:
import torch.nn.functional as F
from transformers import SegformerForSemanticSegmentation

device = "cuda" if torch.cuda.is_available() else "cpu"

model = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/segformer-b2-finetuned-ade-512-512",
    num_labels=1,
    ignore_mismatched_sizes=True
)
model.to(device)

Some weights of SegformerForSemanticSegmentation were not initialized from the model checkpoint at nvidia/segformer-b2-finetuned-ade-512-512 and are newly initialized because the shapes did not match:
- decode_head.classifier.weight: found shape torch.Size([150, 768, 1, 1]) in the checkpoint and torch.Size([1, 768, 1, 1]) in the model instantiated
- decode_head.classifier.bias: found shape torch.Size([150]) in the checkpoint and torch.Size([1]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


SegformerForSemanticSegmentation(
  (segformer): SegformerModel(
    (encoder): SegformerEncoder(
      (patch_embeddings): ModuleList(
        (0): SegformerOverlapPatchEmbeddings(
          (proj): Conv2d(3, 64, kernel_size=(7, 7), stride=(4, 4), padding=(3, 3))
          (layer_norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        )
        (1): SegformerOverlapPatchEmbeddings(
          (proj): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
          (layer_norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        )
        (2): SegformerOverlapPatchEmbeddings(
          (proj): Conv2d(128, 320, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
          (layer_norm): LayerNorm((320,), eps=1e-05, elementwise_affine=True)
        )
        (3): SegformerOverlapPatchEmbeddings(
          (proj): Conv2d(320, 512, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=6e-5, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)

# Mixed precision scaler
scaler = torch.amp.GradScaler('cuda', enabled=(device=="cuda"))

# ====== 7. Loss functions: BCE + Dice + Focal ======
import torch.nn as nn

bce_loss_fn = nn.BCEWithLogitsLoss()  # we can add pos_weight if needed

def dice_loss_logits(logits, targets, smooth=1.0):
    """
    logits: (B,1,H,W) raw logits
    targets: (B,1,H,W) float in {0,1}
    dice loss = 1 - dice_coeff
    """
    probs = torch.sigmoid(logits)
    probs = probs.view(probs.shape[0], -1)
    targets = targets.view(targets.shape[0], -1)
    inter = (probs * targets).sum(dim=1)
    union = probs.sum(dim=1) + targets.sum(dim=1)
    dice = (2.0 * inter + smooth) / (union + smooth)
    return 1.0 - dice.mean()

def focal_loss_logits(logits, targets, alpha=0.25, gamma=2.0):
    """
    Binary focal loss implemented on logits.
    """
    bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
    probs = torch.sigmoid(logits)
    p_t = probs * targets + (1 - probs) * (1 - targets)  # pt
    modulating_factor = (1.0 - p_t) ** gamma
    alpha_factor = targets * alpha + (1 - targets) * (1 - alpha)
    loss = alpha_factor * modulating_factor * bce
    return loss.mean()

def combined_loss(logits, targets, w_bce=1.0, w_dice=1.0, w_focal=1.0):
    """
    Weighted sum of BCE, Dice and Focal — adjust weights as desired.
    targets expected shape: (B,1,H,W) float.
    """
    l_bce = bce_loss_fn(logits, targets)
    l_dice = dice_loss_logits(logits, targets)
    l_focal = focal_loss_logits(logits, targets)
    return w_bce * l_bce + w_dice * l_dice + w_focal * l_focal, (l_bce, l_dice, l_focal)

# ====== 8. Metrics helpers ======
def threshold_predictions(logits, thr=0.5):
    probs = torch.sigmoid(logits)
    return (probs > thr).int()

def compute_batch_metrics(preds, targets):
    """
    preds, targets: (B,H,W) int or long 0/1
    returns: dict metrics averaged over batch
    """
    preds = preds.view(preds.shape[0], -1)
    targets = targets.view(targets.shape[0], -1)
    tp = (preds * targets).sum(dim=1).float()
    fp = (preds * (1 - targets)).sum(dim=1).float()
    fn = ((1 - preds) * targets).sum(dim=1).float()
    tn = ((1 - preds) * (1 - targets)).sum(dim=1).float()

    eps = 1e-6
    precision = (tp / (tp + fp + eps)).mean().item()
    recall = (tp / (tp + fn + eps)).mean().item()
    f1 = (2 * precision * recall / (precision + recall + eps))
    accuracy = ((tp + tn) / (tp + tn + fp + fn + eps)).mean().item()
    iou = (tp / (tp + fp + fn + eps)).mean().item()

    return {"precision": precision, "recall": recall, "f1": f1, "accuracy": accuracy, "iou": iou}

In [ ]:
from tqdm.auto import tqdm

EPOCHS = 40
best_val_iou = 0.0
save_path = "/content/drive/MyDrive/segformer_slum_optimized_best.pth"

for epoch in range(EPOCHS):
    model.train()
    train_losses = []
    train_bce = []
    train_dice = []
    train_focal = []
    train_metrics = {"precision": [], "recall": [], "f1": [], "accuracy": [], "iou": []}

    pbar = tqdm(train_loader, desc=f"Train Epoch {epoch+1}/{EPOCHS}")
    for imgs, masks in pbar:
        imgs = imgs.to(device)
        masks_in = masks.unsqueeze(1).float().to(device)  # (B,1,H,W) float for loss
        masks_eval = masks.to(device)  # (B,H,W) long for metrics

        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=(device=="cuda")):
            outputs = model(pixel_values=imgs)
            logits = outputs.logits  # (B,1,H',W') often matches input size in segformer but be safe
            # if logits not same size as TILE_SIZE, we upsample:
            if logits.shape[-2:] != (TILE_SIZE, TILE_SIZE):
                logits = F.interpolate(logits, size=(TILE_SIZE, TILE_SIZE), mode='bilinear', align_corners=False)

            loss, (l_bce, l_dice, l_focal) = combined_loss(logits, masks_in, w_bce=1.0, w_dice=1.0, w_focal=1.0)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_losses.append(loss.item())
        train_bce.append(l_bce.item())
        train_dice.append(l_dice.item())
        train_focal.append(l_focal.item())

        preds = threshold_predictions(logits, thr=0.5).squeeze(1).cpu()
        batch_metrics = compute_batch_metrics(preds, masks_eval.cpu())
        for k,v in batch_metrics.items():
            train_metrics[k].append(v)

        pbar.set_postfix(loss=np.mean(train_losses), iou=np.mean(train_metrics["iou"]))

    # Aggregate training metrics
    train_summary = {k: float(np.mean(v)) for k,v in train_metrics.items()}
    train_loss_avg = float(np.mean(train_losses))

    # Validation
    model.eval()
    val_losses = []
    val_metrics = {"precision": [], "recall": [], "f1": [], "accuracy": [], "iou": []}
    with torch.no_grad():
        for imgs, masks in tqdm(val_loader, desc=f"Val Epoch {epoch+1}/{EPOCHS}"):
            imgs = imgs.to(device)
            masks_in = masks.unsqueeze(1).float().to(device)
            masks_eval = masks.to(device)

            outputs = model(pixel_values=imgs)
            logits = outputs.logits
            if logits.shape[-2:] != (TILE_SIZE, TILE_SIZE):
                logits = F.interpolate(logits, size=(TILE_SIZE, TILE_SIZE), mode='bilinear', align_corners=False)
            loss, _ = combined_loss(logits, masks_in, w_bce=1.0, w_dice=1.0, w_focal=1.0)
            val_losses.append(loss.item())

            preds = threshold_predictions(logits, thr=0.5).squeeze(1).cpu()
            batch_metrics = compute_batch_metrics(preds, masks_eval.cpu())
            for k,v in batch_metrics.items():
                val_metrics[k].append(v)

    val_summary = {k: float(np.mean(v)) for k,v in val_metrics.items()}
    val_loss_avg = float(np.mean(val_losses))

    # Scheduler step on metric (use val_iou)
    scheduler.step(val_summary["iou"])

    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss {train_loss_avg:.4f} | Val Loss {val_loss_avg:.4f}")
    print("Train Metrics:", train_summary)
    print("Val Metrics:  ", val_summary)

    # Save best model by val IoU
    if val_summary["iou"] > best_val_iou:
        best_val_iou = val_summary["iou"]
        torch.save(model.state_dict(), save_path)
        print(f"Saved best model with val IoU: {best_val_iou:.4f}")

# Also save final checkpoint
torch.save(model.state_dict(), "/content/drive/MyDrive/segformer_slum_final.pth")
print("Training complete. Best val IoU:", best_val_iou)


In [ ]:
# Load best model
model.load_state_dict(torch.load(save_path, map_location=device))
model.to(device)
model.eval()

test_metrics = {"precision": [], "recall": [], "f1": [], "accuracy": [], "iou": []}
with torch.no_grad():
    for imgs, masks in tqdm(test_loader, desc="Testing"):
        imgs = imgs.to(device)
        masks_eval = masks.to(device)
        outputs = model(pixel_values=imgs)
        logits = outputs.logits
        if logits.shape[-2:] != (TILE_SIZE, TILE_SIZE):
            logits = F.interpolate(logits, size=(TILE_SIZE, TILE_SIZE), mode='bilinear', align_corners=False)

        preds = threshold_predictions(logits, thr=0.5).squeeze(1).cpu()
        batch_metrics = compute_batch_metrics(preds, masks_eval.cpu())
        for k,v in batch_metrics.items():
            test_metrics[k].append(v)

test_summary = {k: float(np.mean(v)) for k,v in test_metrics.items()}
print("Test Metrics:", test_summary)


In [ ]:
import torch
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
import torch.nn.functional as F

# Load the best saved model
best_model_path = "/content/drive/MyDrive/segformer_slum_final.pth"  # change if needed
model.load_state_dict(torch.load(best_model_path, map_location=device))
model.eval()  # set to eval mode

# Create test DataLoader
test_dataset = SlumDataset(test_tiles)  # your dataset class
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        logits = outputs.logits
        # Resize logits to TILE_SIZE if necessary
        if logits.shape[-2:] != (TILE_SIZE, TILE_SIZE):
            logits = F.interpolate(logits, size=(TILE_SIZE, TILE_SIZE), mode='bilinear', align_corners=False)

        # Assuming sigmoid for binary segmentation
        preds = (torch.sigmoid(logits) > 0.5).float()

        all_preds.append(preds.cpu())
        all_labels.append(labels.cpu())

# Concatenate all batches
all_preds = torch.cat(all_preds, dim=0)
all_labels = torch.cat(all_labels, dim=0)

def iou_metric(preds, labels, eps=1e-6):
    intersection = (preds * labels).sum(dim=(1,2,3))
    union = (preds + labels - preds*labels).sum(dim=(1,2,3))
    iou = (intersection + eps) / (union + eps)
    return iou.mean().item()

def f1_metric(preds, labels, eps=1e-6):
    TP = (preds * labels).sum(dim=(1,2,3))
    FP = (preds * (1 - labels)).sum(dim=(1,2,3))
    FN = ((1 - preds) * labels).sum(dim=(1,2,3))
    f1 = (2*TP + eps) / (2*TP + FP + FN + eps)
    return f1.mean().item()

test_iou = iou_metric(all_preds, all_labels)
test_f1 = f1_metric(all_preds, all_labels)

print(f"Test IoU: {test_iou:.4f}")
print(f"Test F1: {test_f1:.4f}")

num_examples = 50
fig, axs = plt.subplots(num_examples, 3, figsize=(12, num_examples*4))
for i in range(num_examples):
    img = test_dataset[i][0].permute(1,2,0).numpy()  # convert to HWC
    # Unnormalize image for plotting
    img = img * np.array(STD) + np.array(MEAN)
    gt = test_dataset[i][1].numpy().squeeze()
    pred = all_preds[i].numpy().squeeze()

    axs[i,0].imshow(img)
    axs[i,0].set_title("Image")
    axs[i,1].imshow(gt, cmap='gray')
    axs[i,1].set_title("Ground Truth")
    axs[i,2].imshow(pred, cmap='gray')
    axs[i,2].set_title("Prediction")

    for ax in axs[i]:
        ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import glob
import math

def predict_multiple_custom_images(input_folder, model, device, threshold=0.5, tile_size=TILE_SIZE, stride=STRIDE):
    image_paths = sorted(glob.glob(os.path.join(input_folder, "*.tif")))
    if not image_paths:
        print(f"No .tif files found in {input_folder}")
        return

    mean = np.array(MEAN).reshape(3,1,1)
    std = np.array(STD).reshape(3,1,1)

    model.eval()
    for img_path in image_paths:
        print("Processing:", os.path.basename(img_path))
        with rasterio.open(img_path) as src:
            image = src.read()
            profile = src.profile
        image = select_rgb_bands(image)
        H, W = image.shape[1], image.shape[2]
        img_tensor = image.astype(np.float32) / 255.0
        img_norm = (img_tensor - mean) / std

        prob_accum = np.zeros((H, W), dtype=np.float32)
        weight_accum = np.zeros((H, W), dtype=np.float32)

        ys = list(range(0, max(1, H - tile_size + 1), stride))
        xs = list(range(0, max(1, W - tile_size + 1), stride))
        if ys[-1] + tile_size < H:
            ys.append(H - tile_size)
        if xs[-1] + tile_size < W:
            xs.append(W - tile_size)

        with torch.no_grad():
            for y in ys:
                for x in xs:
                    tile = img_norm[:, y:y+tile_size, x:x+tile_size]
                    t_h, t_w = tile.shape[1], tile.shape[2]
                    if t_h < tile_size or t_w < tile_size:
                        tile = np.pad(tile, ((0,0),(0,tile_size-t_h),(0,tile_size-t_w)), mode='constant')

                    tile_t = torch.from_numpy(tile).float().unsqueeze(0).to(device)
                    logits = model(pixel_values=tile_t).logits
                    logits = F.interpolate(logits, size=(tile_size, tile_size), mode='bilinear', align_corners=False)
                    prob = torch.sigmoid(logits).squeeze().cpu().numpy()  # (H_t, W_t) if squeezed
                    prob = prob[:t_h, :t_w]

                    prob_accum[y:y+t_h, x:x+t_w] += prob
                    weight_accum[y:y+t_h, x:x+t_w] += 1.0

        # avoid division by zero
        weight_accum[weight_accum == 0] = 1.0
        avg_prob = prob_accum / weight_accum
        binary_mask = (avg_prob > threshold).astype(np.uint8)

        # visualization (like your original)
        img_plot = np.transpose(image, (1,2,0)) / 255.0
        img_plot = np.clip(img_plot, 0, 1)

        fig, ax = plt.subplots(1,2, figsize=(12,6))
        ax[0].imshow(img_plot)
        ax[0].set_title(os.path.basename(img_path))
        ax[0].axis('off')
        ax[1].imshow(binary_mask, cmap='gray', vmin=0, vmax=1)
        ax[1].set_title("Predicted Slum (White)")
        ax[1].axis('off')
        plt.tight_layout()
        plt.show()

# Use same input folder variable as you used before
INPUT_FOLDER_PATH = "/images input"
if not os.path.exists(INPUT_FOLDER_PATH):
    os.makedirs(INPUT_FOLDER_PATH)
    print(f"Please upload your .tif images to {INPUT_FOLDER_PATH}")
else:
    predict_multiple_custom_images(INPUT_FOLDER_PATH, model, device, threshold=0.5)